# Gans Pipeline

## Libraries and Constants

In [ ]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

In [ ]:
headers = {'User-Agent': 'Chrome/134.0.0.0'}
load_dotenv()
connection_string = os.getenv("CON_STRING")

## Define Functions

### Cities

#### Basic Data

In [ ]:
def get_cities_info(cities):
    countries = []
    latitudes = []
    longitudes = []

    for city in cities:
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            for row in city_soup.find("table").find_all("tr"):
                if row.find(string="Country"):
                    countries.append(row.find("a").get_text())
            lat, lon = city_soup.find("table").find(class_="geo").get_text().split("; ")
            try:
                latitudes.append(float(lat))
                longitudes.append(float(lon))
            except ValueError:
                latitudes.append(None)
                longitudes.append(None)

        else:
            print(f"WARNING: Could not retrieve HTML for {city}")

    cities_df = pd.DataFrame({"name": cities, "country": countries, "latitude": latitudes, "longitude": longitudes})

    return cities_df

#### Population

In [ ]:
def get_populations(cities_df):
    populations = []
    city_ids = []

    for i, row in cities_df.iterrows():
        city = row["name"]
        city_id = row["city_id"]
        
        url = f"https://en.wikipedia.org/wiki/{city}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            city_soup = BeautifulSoup(response.content, 'html.parser')
            for row in city_soup.find("table").find_all("tr"):
                if row.find(string="Population"):
                    population = int(row.find_next("td").get_text().replace(",", ""))
        populations.append(population)
        city_ids.append(city_id)

    pop_df = pd.DataFrame({"city_id": city_ids, "population": populations, "date_gathered": pd.Timestamp.now().date()})

    return pop_df

## Add new cities

In [ ]:
cities = ["Berlin", "Hamburg", "Munich"]

cities_df = get_cities_info(cities)
cities_df.to_sql(
    "cities",
    con=connection_string,
    if_exists="append",
    index=False
)

In [ ]:
cities_df = pd.read_sql(
    "SELECT * FROM cities WHERE city_id NOT IN (SELECT DISTINCT city_id FROM populations);", 
    con=connection_string
)
pop_df = get_populations(cities_df)
pop_df.to_sql(
    "populations",
    con=connection_string,
    if_exists="append",
    index=False
)